## Setup: Load the dataset

In [ ]:
import pandas as pd
import numpy as np

# For this lab, we'll create a synthetic dataset matching the course examples
np.random.seed(42)

data = {
    'customer_id': ['C-0041', 'C-0042', 'c0043', 'C-0044', 'C-0045', 'C-0041'] + ['C-{0:04d}'.format(i) for i in range(46, 4820)],
    'region': ['APAC', 'EMEA', 'N/A', 'APAC', 'LATAM', 'APAC'] + np.random.choice(['APAC', 'EMEA', 'LATAM', 'NA', 'N/A'], 4814).tolist(),
    'support_tickets': [2.0, np.nan, 0.0, 5.0, 1.0, 2.0] + np.random.randint(0, 10, 4814).astype(float).tolist(),
}

df = pd.DataFrame(data)

# Normalize: add 269 more N/A values to reach 275 total (275 = 6 + 269)
mask = np.random.choice(df.index[6:], 269, replace=False)
df.loc[mask, 'region'] = 'N/A'

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head(10))

## Bug 1: Silent string-null miss

The audit reports 0 missing in `region`, but many rows contain the string `'N/A'`. Find the bug and fix it.

In [ ]:
# --- BUGGY CODE ---
missing = df.isnull().sum()
print(f"Missing values in region: {missing['region']}")
print(f"\nTotal missing summary:")
print(missing)

**Detect the bug:** Run the cell below to find what `isnull()` is missing:

In [ ]:
# Detect: value_counts with dropna=False shows string nulls
print(df['region'].value_counts(dropna=False))
print(f"\nRows with the string 'N/A': {(df['region'] == 'N/A').sum()}")

**Explanation:** Write your answer here — what signal does the audit miss by not replacing string nulls first?


**Fix the bug:**

In [ ]:
# Step 1: Replace string nulls with np.nan
df['region'] = df['region'].replace('N/A', np.nan)

# Step 2: Rerun the audit
missing = df.isnull().sum()
print(f"Missing values in region after fix: {missing['region']}")

## Bug 2: Casing not normalised before counting unique values

The audit reports 4832 unique customer IDs. But there's a casing issue: 'C-0043' and 'c0043'. Find the bug.

In [ ]:
# --- BUGGY CODE ---
print(f"Unique customer IDs (uncased): {df['customer_id'].nunique()}")

**Detect the bug:**

In [ ]:
print(f"Unique customer IDs (after str.upper()): {df['customer_id'].str.upper().nunique()}")
print(f"Difference: {df['customer_id'].nunique() - df['customer_id'].str.upper().nunique()}")

**Explanation:** Write your answer here — what downstream impact would this have on features that count unique customers?

**Fix the bug:**

In [ ]:
# Normalize casing before counting
df['customer_id'] = df['customer_id'].str.upper()
print(f"Unique customer IDs after normalization: {df['customer_id'].nunique()}")

## Bug 3: Full-row deduplication misses logical duplicates

The audit reports 0 duplicate rows. But some customer IDs appear more than once. Find the bug.

In [ ]:
# --- BUGGY CODE ---
print(f"Full-row duplicates: {df.duplicated().sum()}")

**Detect the bug:**

In [ ]:
# Check for duplicates on the key column
print(f"Duplicate customer IDs: {df.duplicated(subset=['customer_id']).sum()}")
print(f"\nCustomer C-0041 appears {(df['customer_id'] == 'C-0041').sum()} times")

**Explanation:** Write your answer here — why is `drop_duplicates()` without a `subset` parameter insufficient?

**Fix the bug:**

In [ ]:
# Always check for key-level duplicates after full-row deduplication
print(f"Key-level duplicates (subset=['customer_id']): {df.duplicated(subset=['customer_id']).sum()}")

# If key-level duplicates exist, decide which row to keep
# For this example, keep the first occurrence per customer
df_clean = df.drop_duplicates(subset=['customer_id'], keep='first')
print(f"\nAfter deduplication on key: {len(df_clean)} rows (was {len(df)})")

## Summary check

Verify all three bugs have been fixed:

In [ ]:
# After all fixes, run the full audit
def audit(df):
    print(f"Shape: {df.shape}")
    print(f"\nMissing values:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    print(pd.DataFrame({'count': missing, 'pct': missing_pct}).sort_values('pct', ascending=False))
    print(f"\nUnique customer IDs: {df['customer_id'].nunique()}")
    print(f"Full duplicates: {df.duplicated().sum()}")
    print(f"Key-level duplicates: {df.duplicated(subset=['customer_id']).sum()}")

audit(df_clean)